# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Races Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType

races_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("race_name", StringType(), True),
    StructField("url", StringType(), True),
    StructField("date", DateType(), True),
    StructField("time", StringType(), True),
    StructField("circuit_id", StringType(), False),
    StructField("circuit_name", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("locality", StringType(), True),
    StructField("country", StringType(), True),
    StructField("first_practice_date", StringType(), True),
    StructField("first_practice_time", StringType(), True),
    StructField("second_practice_date", StringType(), True),
    StructField("second_practice_time", StringType(), True),
    StructField("third_practice_date", StringType(), True),
    StructField("third_practice_time", StringType(), True),
    StructField("qualifying_date", StringType(), True),
    StructField("qualifying_time", StringType(), True),
    StructField("sprint_date", StringType(), True),
    StructField("sprint_time", StringType(), True),
])

races_input_path = f"{processed_folder_path}/races/csv/races.csv"

races_df = spark.read \
    .option("header", True) \
    .schema(races_schema) \
    .csv(races_input_path)


# 3) Transform Races Data:

The steps included:

- Drop column "url", "circuit_name", "lat", "long", "locality", "country".
- Create Surrogate Key.
- Add Data Source and File Date.
- Fill Null cells with "None"

In [0]:
from pyspark.sql.functions import lit

races_with_audit_df = races_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

races_date_df = add_ingestion_date(races_with_audit_df)
races_fill_df = races_date_df.fillna("None")
races_dropped_df = races_fill_df.drop("url", "circuit_name", "lat", "long", "locality", "country")

races_final_df = add_surrogate_key(
    races_dropped_df,
    key_column_name="races_sk",
    hash_columns=["season", "round", "race_name", "date", "time", "circuit_id", "first_practice_date", "first_practice_time", "second_practice_date", "second_practice_time", "third_practice_date", "third_practice_time","qualifying_date", "qualifying_time", "sprint_date", "sprint_time"],
)

print("Final columns going into the write:", races_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
races_output_path = f"{processed_folder_path}/races/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=races_final_df,
    db_name="f1_processed",
    table_name="races",
    output_path=races_output_path,
    merge_key_columns=["season", "round"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(races_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/races/delta",
    presentation_directory=f"{presentation_folder_path}/fact_races/delta",
    db_name="f1_presentation",
    table_name="fact_races",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_races/delta"))

# 5) Save backup Races in CSV format:

In [0]:
import io
import csv

races_backup_path = f"{presentation_folder_path}/fact_races/csv/fact_races.csv"

backup_rows = [row.asDict() for row in races_final_df.collect()]
backup_fieldnames = races_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(races_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {races_backup_path}")